# Import Library & Download NLTK

In [1]:
!pip install gensim

import pandas as pd
import numpy as np
import re
import nltk
from nltk.corpus import stopwords
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from gensim.models import Word2Vec

import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout, Bidirectional
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

nltk.download('stopwords')
print("Library berhasil di-import.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 42.7 MB/s eta 0:00:00
Library berhasil di-import.


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


# Load Data & Preprocessing Teks

In [3]:
# 1. Load dataset
df_raw = pd.read_csv('dataset_sentimen.csv')

# 2. Kamus Slang yang diperluas untuk menangkap arti kata gaul/singkatan di Play Store
kamus_slang = {
    "yg": "yang", "gbr": "gambar", "gk": "tidak", "gak": "tidak", "ga": "tidak",
    "g": "tidak", "tdk": "tidak", "krn": "karena", "bgt": "banget", "apk": "aplikasi",
    "dgn": "dengan", "aja": "saja", "lemot": "lambat", "lola": "lambat", "bgst": "jelek",
    "mantap": "bagus", "ok": "bagus", "oke": "bagus", "top": "bagus", "can": "bisa",
    "tp": "tetapi", "sdh": "sudah", "udh": "sudah", "dah": "sudah", "bisaa": "bisa",
    "please": "tolong", "pls": "tolong", "apps": "aplikasi", "error": "rusak", "nyesel": "kecewa"
}
stop_words = set(stopwords.words('indonesian'))

def bersihkan_teks(text):
    text = str(text).lower()                 # Case folding
    text = re.sub(r'[^a-zA-Z\s]', '', text)  # Hapus angka & tanda baca
    words = text.split()
    # Normalisasi slang dan buang stopword
    words_cleaned = [kamus_slang.get(w, w) for w in words if w not in stop_words]
    return " ".join(words_cleaned)

df_raw['text_clean'] = df_raw['content'].apply(bersihkan_teks)

# 3. Filter Kebisingan: Buang teks yang terlalu pendek (di bawah 4 kata) karena biasanya bias
df_filtered = df_raw[df_raw['text_clean'].apply(lambda x: len(x.split()) >= 4)]

# 4. Balancing Data: Ambil sampel sama rata untuk 3 kelas agar tidak bias
min_size = df_filtered['label'].value_counts().min()
target_per_kelas = max(min_size, 3400) # Memastikan total data akhir tetap di atas 10.000 sampel

df_balanced = pd.DataFrame()
for kelas in ['negatif', 'netral', 'positif']:
    df_kelas = df_filtered[df_filtered['label'] == kelas]
    if len(df_kelas) >= target_per_kelas:
        df_resampled = df_kelas.sample(n=target_per_kelas, random_state=42, replace=False)
    else:
        df_resampled = df_kelas.sample(n=target_per_kelas, random_state=42, replace=True)
    df_balanced = pd.concat([df_balanced, df_resampled], axis=0)

# Acak ulang urutan data
df = df_balanced.sample(frac=1, random_state=42).reset_index(drop=True)

# 5. Encode Label Kategori
le = LabelEncoder()
df['label_encoded'] = le.fit_transform(df['label'])

print(f"Total data bersih siap pakai: {len(df)} sampel")
print("\nJumlah data per kelas (Seimbang):")
print(df['label'].value_counts())

Total data bersih siap pakai: 10200 sampel

Jumlah data per kelas (Seimbang):
label
positif    3400
netral     3400
negatif    3400
Name: count, dtype: int64


# Persiapan Tokenizer & Ekstraksi Fitur Dasar

In [4]:
MAX_WORDS = 15000  # Dinaikkan dari 10.000 agar menampung kosakata lebih luas
MAX_LEN = 100

tokenizer = Tokenizer(num_words=MAX_WORDS, oov_token="<OOV>")
tokenizer.fit_on_texts(df['text_clean'])

X_sequences = tokenizer.texts_to_sequences(df['text_clean'])
X_padded = pad_sequences(X_sequences, maxlen=MAX_LEN, padding='post', truncating='post')
y = tf.keras.utils.to_categorical(df['label_encoded'], num_classes=3)

# Global Callbacks untuk menghentikan pelatihan secara cerdas dan menghindari overfitting
callbacks_list = [
    EarlyStopping(monitor='val_accuracy', patience=3, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=2, min_lr=1e-5, verbose=1)
]
print("Persiapan fitur teks selesai.")

Persiapan fitur teks selesai.


# SKEMA 1 (LSTM + Keras Embedding + Split 80/20)

In [5]:
X_train1, X_test1, y_train1, y_test1 = train_test_split(X_padded, y, test_size=0.2, random_state=42)

model1 = Sequential([
    Embedding(input_dim=MAX_WORDS, output_dim=128, input_length=MAX_LEN),
    LSTM(128, return_sequences=True),
    tf.keras.layers.GlobalMaxPooling1D(),
    Dense(64, activation='relu'),
    Dropout(0.3),
    Dense(3, activation='softmax')
])

model1.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

history1 = model1.fit(
    X_train1,
    y_train1,
    epochs=12,
    batch_size=32,
    validation_data=(X_test1, y_test1),
    callbacks=callbacks_list
)

Epoch 1/12


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


255/255 ━━━━━━━━━━━━━━━━━━━━ 54s 201ms/step - accuracy: 0.6582 - loss: 0.7611 - val_accuracy: 0.8225 - val_loss: 0.4643 - learning_rate: 0.0010
Epoch 2/12
255/255 ━━━━━━━━━━━━━━━━━━━━ 49s 193ms/step - accuracy: 0.8874 - loss: 0.3196 - val_accuracy: 0.8755 - val_loss: 0.3374 - learning_rate: 0.0010
Epoch 3/12
255/255 ━━━━━━━━━━━━━━━━━━━━ 80s 187ms/step - accuracy: 0.9444 - loss: 0.1709 - val_accuracy: 0.8961 - val_loss: 0.3076 - learning_rate: 0.0010
Epoch 4/12
255/255 ━━━━━━━━━━━━━━━━━━━━ 50s 195ms/step - accuracy: 0.9683 - loss: 0.1011 - val_accuracy: 0.9005 - val_loss: 0.3502 - learning_rate: 0.0010
Epoch 5/12
255/255 ━━━━━━━━━━━━━━━━━━━━ 0s 175ms/step - accuracy: 0.9778 - loss: 0.0725
Epoch 5: ReduceLROnPlateau reducing learning rate to 0.00020000000949949026.
255/255 ━━━━━━━━━━━━━━━━━━━━ 49s 190ms/step - accuracy: 0.9800 - loss: 0.0660 - val_accuracy: 0.9069 - val_loss: 0.3562 - learning_rate: 0.0010
Epoch 6/12
255/255 ━━━━━━━━━━━━━━━━━━━━ 48s 188ms/step - accuracy: 0.9892 - loss: 

# SKEMA 2 (Bi-LSTM + Keras Embedding + Split 80/20)

In [6]:
X_train2, X_test2, y_train2, y_test2 = train_test_split(X_padded, y, test_size=0.2, random_state=42)

model2 = Sequential([
    Embedding(input_dim=MAX_WORDS, output_dim=256, input_length=MAX_LEN),
    Bidirectional(LSTM(128, return_sequences=True, dropout=0.3, recurrent_dropout=0.3)),
    Bidirectional(LSTM(64, dropout=0.3, recurrent_dropout=0.3)),
    Dense(64, activation='relu'),
    Dropout(0.5),
    Dense(3, activation='softmax')
])

model2.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

history2 = model2.fit(
    X_train2,
    y_train2,
    epochs=12,
    batch_size=32,
    validation_data=(X_test2, y_test2),
    callbacks=callbacks_list
)

Epoch 1/12
255/255 ━━━━━━━━━━━━━━━━━━━━ 274s 1s/step - accuracy: 0.6266 - loss: 0.7984 - val_accuracy: 0.8176 - val_loss: 0.4775 - learning_rate: 0.0010
Epoch 2/12
255/255 ━━━━━━━━━━━━━━━━━━━━ 0s 943ms/step - accuracy: 0.8702 - loss: 0.3724
Epoch 2: ReduceLROnPlateau reducing learning rate to 0.00020000000949949026.
255/255 ━━━━━━━━━━━━━━━━━━━━ 251s 984ms/step - accuracy: 0.8817 - loss: 0.3488 - val_accuracy: 0.8770 - val_loss: 0.3437 - learning_rate: 0.0010
Epoch 3/12
255/255 ━━━━━━━━━━━━━━━━━━━━ 271s 1s/step - accuracy: 0.9521 - loss: 0.1640 - val_accuracy: 0.8941 - val_loss: 0.3377 - learning_rate: 2.0000e-04
Epoch 3: early stopping
Restoring model weights from the end of the best epoch: 1.


# SKEMA 3 (Bi-LSTM + Word2Vec Embedding + Split 70/30)

In [8]:
# Split data baru 70/30 (Teks mentah)
X_train3_text, X_test3_text, y_train3, y_test3 = train_test_split(df['text_clean'], y, test_size=0.3, random_state=42)

# Latih Word2Vec lokal dengan parameter representasi tinggi
sentences = [sentence.split() for sentence in X_train3_text]
w2v_model = Word2Vec(sentences, vector_size=256, window=5, min_count=1, workers=4)

# Bangun Embedding Matrix dari Word2Vec
embedding_matrix = np.zeros((MAX_WORDS, 256))
for word, i in tokenizer.word_index.items():
    if i < MAX_WORDS and word in w2v_model.wv:
        embedding_matrix[i] = w2v_model.wv[word]
    elif i < MAX_WORDS:
        # Jika kata tidak ada di Word2Vec, isi dengan nilai acak kecil (bukan 0 mutlak)
        embedding_matrix[i] = np.random.normal(0, 0.1, 256)

X_train3_seq = tokenizer.texts_to_sequences(X_train3_text)
X_test3_seq = tokenizer.texts_to_sequences(X_test3_text)

X_train3_padded = pad_sequences(X_train3_seq, maxlen=MAX_LEN, padding='post', truncating='post')
X_test3_padded = pad_sequences(X_test3_seq, maxlen=MAX_LEN, padding='post', truncating='post')

# Bangun Arsitektur Model 3
model3 = Sequential([
    Embedding(input_dim=MAX_WORDS, output_dim=256, weights=[embedding_matrix], input_length=MAX_LEN, trainable=True),
    Bidirectional(LSTM(64, dropout=0.2)),
    Dense(32, activation='relu'),
    Dropout(0.4),
    Dense(3, activation='softmax')
])

model3.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

# Jalankan Pelatihan dengan data yang sudah di-padded yang benar
history3 = model3.fit(
    X_train3_padded, # Gunakan data padded yang baru dibuat
    y_train3,
    epochs=12,
    batch_size=32,
    validation_data=(X_test3_padded, y_test3), # Gunakan data padded yang baru dibuat
    callbacks=callbacks_list
)

Epoch 1/12
224/224 ━━━━━━━━━━━━━━━━━━━━ 70s 289ms/step - accuracy: 0.4759 - loss: 0.9805 - val_accuracy: 0.6039 - val_loss: 0.7817 - learning_rate: 0.0010
Epoch 2/12
224/224 ━━━━━━━━━━━━━━━━━━━━ 0s 265ms/step - accuracy: 0.6918 - loss: 0.6659
Epoch 2: ReduceLROnPlateau reducing learning rate to 0.00020000000949949026.
224/224 ━━━━━━━━━━━━━━━━━━━━ 70s 312ms/step - accuracy: 0.7606 - loss: 0.5637 - val_accuracy: 0.8405 - val_loss: 0.4005 - learning_rate: 0.0010
Epoch 3/12
224/224 ━━━━━━━━━━━━━━━━━━━━ 76s 284ms/step - accuracy: 0.9099 - loss: 0.2760 - val_accuracy: 0.8686 - val_loss: 0.3564 - learning_rate: 2.0000e-04
Epoch 3: early stopping
Restoring model weights from the end of the best epoch: 1.


# Evaluasi Akhir

In [9]:
print("Perbandingan Akurasi Akhir di Testing Set")
print(f"Akurasi Testing Skema 1: {history1.history['val_accuracy'][-1]*100:.2f}%")
print(f"Akurasi Testing Skema 2: {history2.history['val_accuracy'][-1]*100:.2f}%")
print(f"Akurasi Testing Skema 3: {history3.history['val_accuracy'][-1]*100:.2f}%")

Perbandingan Akurasi Akhir di Testing Set
Akurasi Testing Skema 1: 91.62%
Akurasi Testing Skema 2: 89.41%
Akurasi Testing Skema 3: 86.86%


# Inference

In [10]:
def prediksi_sentimen(teks_baru, model_pilihan):
    # 1. Bersihkan teks menggunakan fungsi yang sudah dibuat di Cell 2
    teks_bersih = bersihkan_teks(teks_baru)

    # 2. Tokenisasi teks menjadi sekuens angka
    sekuens = tokenizer.texts_to_sequences([teks_bersih])

    # 3. Lakukan padding agar panjangnya sesuai (MAX_LEN = 100)
    padded = pad_sequences(sekuens, maxlen=MAX_LEN, padding='post', truncating='post')

    # 4. Lakukan prediksi probabilitas kelas
    prediksi = model_pilihan.predict(padded, verbose=0)

    # 5. Ambil indeks kelas dengan probabilitas tertinggi
    kelas_id = np.argmax(prediksi)

    # 6. Kembalikan ke label teks asli (negatif/netral/positif)
    label_sentimen = le.inverse_transform([kelas_id])[0]

    # Ambil nilai probabilitas untuk tingkat keyakinan model (opsional untuk mempercantik output)
    tingkat_keyakinan = prediksi[0][kelas_id] * 100

    return label_sentimen, tingkat_keyakinan

# List ulasan acak baru yang belum pernah dilihat oleh model sebelumnya
ulasan_tes = [
    "Aplikasi apaan sih ini? Tiap mau bayar selalu error dan keluar sendiri. Gak guna bgt!",
    "Layanannya lumayan oke, tapi tolong diperbaiki lagi loading gambarnya yang agak lambat.",
    "Wah mantap banget! Transaksinya instan, fiturnya lengkap, dan tampilannya ramah pengguna."
]

for i, teks in enumerate(ulasan_tes, 1):
    hasil_kelas, keyakinan = prediksi_sentimen(teks, model1)

    print(f"Uji Coba #{i}")
    print(f"Teks Input : \"{teks}\"")
    print(f"Output     : SENTIMEN {hasil_kelas.upper()} ({keyakinan:.2f}% yakin)")

Uji Coba #1
Teks Input : "Aplikasi apaan sih ini? Tiap mau bayar selalu error dan keluar sendiri. Gak guna bgt!"
Output     : SENTIMEN NEGATIF (55.76% yakin)
Uji Coba #2
Teks Input : "Layanannya lumayan oke, tapi tolong diperbaiki lagi loading gambarnya yang agak lambat."
Output     : SENTIMEN POSITIF (100.00% yakin)
Uji Coba #3
Teks Input : "Wah mantap banget! Transaksinya instan, fiturnya lengkap, dan tampilannya ramah pengguna."
Output     : SENTIMEN POSITIF (99.54% yakin)
